# Silver — `ecommerce_pedidos`

Este notebook lê micro-lotes da Bronze, aplica as 10 regras de qualidade da tabela de pedidos, grava a Silver em Delta e registra os resultados em `squad1.dq_monitoring_logs`.



In [0]:
# MAGIC %run ../../utils/utils

## Inicialização e Orquestração

In [0]:
import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone
from functools import reduce

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_pedidos"
TABELA_DQ = "dq_monitoring_logs"
DATA_EXECUCAO = datetime.now(timezone.utc)

print(f"Iniciando processamento Silver - Run ID: {RUN_ID}")

## Leitura Dinâmica do Micro-lote e Tabelas de Referência

In [0]:
# 1. Carrega a tabela Bronze de Pedidos
try:
    df_bronze_pedidos = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

# 2. Isola o Micro-lote (Considerando Silver E Quarentena de Pedidos)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_pedido") \
            .union(df_quarentena.select("id_pedido"))
    else:
        df_processados = df_silver_atual.select("id_pedido")
        
    df_micro_lote = df_bronze_pedidos.join(df_processados, "id_pedido", "left_anti")
else:
    df_micro_lote = df_bronze_pedidos

qtd_novos = df_micro_lote.count()
print(f"Registros novos de pedidos para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura das Tabelas de Referência Corretas para Validar Pedidos
# =================================================================================
def obter_referencia(camada, tabela, colunas_select):
    if delta_existe(camada, tabela, STORAGE_OPTIONS):
        return ler_delta(camada, tabela, STORAGE_OPTIONS).select(*colunas_select).dropDuplicates()
    else:
        schema = StructType([StructField(c, StringType(), True) for c in colunas_select])
        return spark.createDataFrame([], schema)

# CORREÇÃO R2/R10: ambas agora usam obter_referencia_silver_ou_bronze,
# priorizando a Silver e caindo para a Bronze quando ela ainda não existir.
df_clientes_ref = obter_referencia_ids_uniao_silver_bronze("ecommerce_clientes", "id_cliente") \
    .withColumnRenamed("id_cliente", "id_cliente_ref")
 
df_enderecos_ref = obter_referencia_silver_ou_bronze("ecommerce_enderecos", ["id_endereco"]) \
    .withColumnRenamed("id_endereco", "id_endereco_ref")
 
# Validação do status de rastreamento logístico (Regra 8)
df_rastreamento_ref = obter_referencia_silver_ou_bronze("ecommerce_rastreamento", ["id_pedido_ecommerce", "status_entrega"]) \
    .withColumnRenamed("id_pedido_ecommerce", "id_pedido_rastreio") \
    .withColumnRenamed("status_entrega", "status_rastreamento")
 
print("Tabelas de referência de Pedidos carregadas com sucesso.")

## Aplicação das 10 Regras de Qualidade (Data Quality)


In [0]:
if qtd_novos > 0:
    from functools import reduce
    
    # Domínios de valores permitidos (Regras 3 e 5)
    status_validos = ["Processando", "Pagamento Aprovado", "Em Separação", "Enviado", "Entregue", "Cancelado"]
    metodos_validos = ["cartão de crédito", "pix", "boleto"] # Deixe em minúsculo se o seu dado for padronizado assim
    
    # Janela para checar duplicidade de PK no próprio lote (Regra 1)
    w_id_pedido = Window.partitionBy("id_pedido")

    # Prepara o DataFrame aplicando os Joins com as tabelas de referência que você carregou
    df_base = df_micro_lote \
        .withColumn("valor_total_num", F.col("valor_total").cast("double")) \
        .withColumn("valor_frete_num", F.coalesce(F.col("valor_frete").cast("double"), F.lit(0.0))) \
        .withColumn("dt_pedido_ts", F.col("dt_pedido").cast("timestamp")) \
        .withColumn("dt_status_ts", F.col("dt_ultima_atualizacao_status").cast("timestamp")) \
        .withColumn("qtd_id_pedido", F.count("*").over(w_id_pedido)) \
        .join(df_clientes_ref, df_micro_lote.id_cliente == df_clientes_ref.id_cliente_ref, "left_outer") \
        .join(df_enderecos_ref, df_micro_lote.id_endereco_entrega == df_enderecos_ref.id_endereco_ref, "left_outer") \
        .join(df_rastreamento_ref, df_micro_lote.id_pedido == df_rastreamento_ref.id_pedido_rastreio, "left_outer")

    # Aplicação massiva de regras
    df_silver_pedidos = df_base \
        .withColumn("r1_id_pedido_falhou", F.col("id_pedido").isNull() | (F.col("id_pedido").cast("string") == "") | (F.col("qtd_id_pedido") > 1)) \
        .withColumn("r2_id_cliente_fk_falhou", F.col("id_cliente").isNull() | F.col("id_cliente_ref").isNull()) \
        .withColumn("r3_status_pedido_falhou", F.col("status_pedido").isNull() | (~F.col("status_pedido").isin(status_validos))) \
        .withColumn("r4_valor_total_falhou", F.col("valor_total_num").isNull() | (F.col("valor_total_num") <= 0)) \
        .withColumn("r5_metodo_pagamento_falhou", F.col("metodo_pagamento").isNull() | (~F.col("metodo_pagamento").isin(metodos_validos))) \
        .withColumn("r6_datas_status_falhou", F.col("dt_pedido_ts").isNull() | F.col("dt_status_ts").isNull() | (F.col("dt_status_ts") < F.col("dt_pedido_ts"))) \
        .withColumn("r7_frete_gratis_falhou", (F.col("valor_total_num") >= 250) & (F.col("valor_frete_num") > 0)) \
        .withColumn("r8_cancelado_rastreamento_falhou", (F.col("status_pedido") == "Cancelado") & F.col("status_rastreamento").isNotNull() & (F.col("status_rastreamento") != "Cancelado")) \
        .withColumn("r9_entrega_tempo_minimo_falhou", (F.col("status_pedido") == "Entregue") & (F.datediff(F.col("dt_status_ts"), F.col("dt_pedido_ts")) < 2)) \
        .withColumn("r10_endereco_fk_falhou", F.col("id_endereco_entrega").isNull() | F.col("id_endereco_ref").isNull()) # CORRIGIDO AQUI!

    # Divisão de Severidade conforme o seu plano (Regras 7 e 9 são AVISOS, o resto trava na quarentena)
    # REGRAS 1-10: Severidade unificada para Critica para garantir o mesmo contrato rígido
    regras_criticas = [
        "r1_id_pedido_falhou", "r2_id_cliente_fk_falhou", "r3_status_pedido_falhou", 
        "r4_valor_total_falhou", "r5_metodo_pagamento_falhou", "r6_datas_status_falhou", 
        "r7_frete_gratis_falhou", "r8_cancelado_rastreamento_falhou", 
        "r9_entrega_tempo_minimo_falhou", "r10_endereco_fk_falhou"
    ]
    
    # Criando a condição lógica unificada usando reduce
    condicao_total_falha = reduce(lambda a, b: a | b, [F.col(c) for c in regras_criticas])

    # Controle final de gravação sem o campo 'silver_tem_aviso'
    df_silver_pedidos = (df_silver_pedidos
        .withColumn("silver_linha_valida", ~condicao_total_falha)
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID)))
    
    print("Muralha de qualidade unificada para Pedidos estruturada. Campo 'silver_tem_aviso' removido.")
else:
    print("Nenhum pedido novo encontrado para processamento.")

## Catálogo de Logs de Pedidos

In [0]:
if qtd_novos > 0:
    from functools import reduce
    
    regras_catalogo = [
        {"coluna": "r1_id_pedido_falhou", "regra": "R1_ID_PEDIDO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_cliente_fk_falhou", "regra": "R2_ID_CLIENTE_FK_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_status_pedido_falhou", "regra": "R3_STATUS_PEDIDO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r4_valor_total_falhou", "regra": "R4_VALOR_TOTAL_INVALIDO", "severidade": "Critica"},
        {"coluna": "r5_metodo_pagamento_falhou", "regra": "R5_METODO_PAGAMENTO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r6_datas_status_falhou", "regra": "R6_DATA_STATUS_ANTERIOR_PEDIDO", "severidade": "Critica"},
        {"coluna": "r7_frete_gratis_falhou", "regra": "R7_FRETE_GRATIS_ACIMA_250", "severidade": "Critica"},          # Modificado para Critica
        {"coluna": "r8_cancelado_rastreamento_falhou", "regra": "R8_CANCELADO_COM_RASTREAMENTO_ATIVO", "severidade": "Critica"},
        {"coluna": "r9_entrega_tempo_minimo_falhou", "regra": "R9_ENTREGA_TEMPO_MINIMO_INVALIDO", "severidade": "Critica"},  # Modificado para Critica
        {"coluna": "r10_endereco_fk_falhou", "regra": "R10_ENDERECO_FK_INVALIDO", "severidade": "Critica"}
    ]

    total_registros = df_silver_pedidos.count()
    logs_list = []

    for r in regras_catalogo:
        qtd_falhas = df_silver_pedidos.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                1, int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    print("Logs calculados de forma unificada. Registros com falhas serão movidos para a Quarentena.")
else:
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Etapa ignorada: não há micro-lote novo.")

# Gravação Final via SDK (Ignorando Unity Catalog)

In [0]:
if qtd_novos > 0:
    # CORREÇÃO: Removido "silver_tem_aviso" do mapeamento de colunas finais da Silver
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]
    
    # ---------------- 1. GRAVAÇÃO DOS VÁLIDOS ---------------- #
    df_silver_validos = (df_silver_pedidos
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_finais))
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=True
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # ---------------- 2. GRAVAÇÃO DA QUARENTENA (DEDUPLICADA) ---------------- #
    df_silver_invalidos = (df_silver_pedidos
        .filter(F.col("silver_linha_valida") == False)
        .select(*colunas_finais))
        
    if df_silver_invalidos.count() > 0:
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            
            # CORREÇÃO CRÍTICA: Alterado de id_cliente para id_pedido para não descartar dados na Quarentena
            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("id_pedido"), 
                on="id_pedido", 
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            sucesso_quarentena = gravar_delta(
                df=df_quarentena_para_gravar,
                camada="silver/quarentena",
                tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS,
                mode="append",
                particionar=False 
            )
            if sucesso_quarentena:
                print(f"Enviados {qtd_novos_rejeitados} registros novos para a quarentena.")
        else:
            print("Todos os registros reprovados já existiam na quarentena histórica.")

    # ---------------- 3. GRAVAÇÃO DOS LOGS NA RAIZ ---------------- #
    if 'df_dq_monitoring_logs_novos' in locals() and df_dq_monitoring_logs_novos.count() > 0:
        # CORREÇÃO: upsert (merge) por tabela+regra+dia em vez de append cego —
        # evita duplicar linhas quando o mesmo lote é reprocessado no mesmo dia
        # (testes, resets). Ver função gravar_dq_logs_upsert() no utils.py.
        sucesso_logs = gravar_dq_logs_upsert(df_dq_monitoring_logs_novos, STORAGE_OPTIONS)
        if sucesso_logs:
            print("Logs de qualidade sincronizados (upsert por tabela+regra+dia)!")
else:
    print("Rotina finalizada sem alterações físicas.")


##  Validação Final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros totais na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))

if delta_existe("silver", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("silver", "dq_monitoring_logs", STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    print(f"Total de violações registradas para {TABELA_ALVO}:", df_logs_validacao.count())
    display(df_logs_validacao.orderBy(F.col("timestamp_execucao").desc()).limit(20))